In [20]:
!pip install elasticsearch --upgrade  -i  https://mirrors.aliyun.com/pypi/simple/ 

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 1.4 MB/s eta 0:00:00a 0:00:01
  Using cached https://mirrors.aliyun.com/pypi/packages/36/4a/6a1cc529ec609dae4d681baddde9ad965b8952d1f56dadbb1d27fbb50ec9/elastic_transport-9.2.0-py3-none-any.whl (65 kB)
  Attempting uninstall: elastic-transport
    Found existing installation: elastic-transport 8.17.1
    Uninstalling elastic-transport-8.17.1:
      Successfully uninstalled elastic-transport-8.17.1
  Attempting uninstall: elasticsearch
    Found existing installation: elasticsearch 8.17.2
    Uninstalling elasticsearch-8.17.2:
      Successfully uninstalled elasticsearch-8.17.2


In [7]:
# 1 构建mapping
index_mapping = {  
    "mappings": {
        "properties": {
            "docContent":  {
                "type": "text",
            },
            "docEmbedding": {
                    "type": "dense_vector",
                    "dims": 1536,
                    "index": True,
                    "similarity": "cosine"
            },
            "title":  {
                "type": "text",
            },
            "dataTime": {
                "type": "text",
            },

        }
    }
}

In [8]:
from elasticsearch import Elasticsearch

# 连接到远程的Elasticsearch实例
es_tool = Elasticsearch(
        hosts=[
            "http://localhost:9200",
        ],
        basic_auth=("elastic", "Dhf8IOJU"),
    verify_certs=True
)

In [9]:
def create_index(es,index_name, index_mapping):
    '''

    :param index_name: 索引名字
    :param index_mapping: 索引结构
    {
        "mappings": {
            "properties": {
                "title": {"type": "text"},
                "description": {"type": "text"},
                "price": {"type": "float"},
                "created_at": {"type": "date"}
            }
        }
    }
    :return:
    '''
    try:
        # 创建索引
        if not es.indices.exists(index=index_name, ):
            es.indices.create(index=index_name, body=index_mapping)
            print(f"Index '{index_name}' CREATED successfully.")
            return True
        else:
            print(f"Index '{index_name}' already exists.")
            return False
    except Exception as e:
        print(f"Index '{index_name}' CREATE failed. error is {str(e)}")
        return False

In [10]:
es_tool.indices.exists(index="test")

HeadApiResponse(True)

In [11]:
create_index(es_tool,"test", index_mapping=index_mapping)

Index 'test' already exists.


False

In [12]:
es_tool.indices.exists(index="test")

HeadApiResponse(True)

In [13]:
es_tool.indices.get_mapping(index="test")

ObjectApiResponse({'test': {'mappings': {'properties': {'dataTime': {'type': 'text'}, 'docContent': {'type': 'text'}, 'docContext': {'type': 'text', 'fields': {'keyword': {'type': 'keyword', 'ignore_above': 256}}}, 'docEmbedding': {'type': 'dense_vector', 'dims': 1536, 'index': True, 'similarity': 'cosine', 'index_options': {'type': 'bbq_hnsw', 'm': 16, 'ef_construction': 100, 'rescore_vector': {'oversample': 3.0}}}, 'title': {'type': 'text'}}}}})

In [14]:
# 写入一条数据
def add_doc(es, index_name, document, id):
    try:
        es.index(index=index_name, id=id, document=document)
        print(f"Index '{index_name}' ,  Doc Id '{id}' ADD successfully.")
        return True
    except Exception as e:
        print(f"Index '{index_name}',  Doc Id '{id}' ADD failed. error is {str(e)}")
        return False

In [15]:
docs = []
with open("doc1.json", "r") as f:
    for line in f:
        docs.append(json.loads(line))

In [16]:
docs[0]

{'subTitle': '',
 'dataTime': '2024-12-24',
 'contentText': '第二十二次全省民政会议召开金湘军出席并讲话\u3000\u3000本报讯（记者张巨峰）12月23日，第二十二次全省民政会议召开，传达学习贯彻习近平总书记对民政工作的重要指示精神和第十五次全国民政会议精神，落实省委要求，安排部署下一步民政工作。省委副书记、省长金湘军出席并讲话。副省长林红玉参加。\u3000\u3000金湘军指出，民政工作连着千家万户，事关百姓福祉。省委、省政府高度重视民政事业发展，近年来，持续在基本民生保障、基本社会服务、基层社会治理等方面下功夫，滚动实施民生实事，民政事业取得新进展新成就。新征程上，要深入贯彻习近平总书记关于民政工作的重要论述和对山西工作的重要讲话重要指示精神，深刻把握民政工作的政治属性、人民立场、职责定位和时代要求，全面落实党中央、国务院决策部署，进一步全面深化改革，加强普惠性、基础性、兜底性民生建设，积极主动为人民群众做好事、办实事、解难事，奋力答好民生答卷，以民政事业高质量发展助力我省现代化建设。\u3000\u3000金湘军就下一步民政工作提出要求。一要积极应对人口老龄化，加快健全养老服务体系。巩固居家养老服务基础地位，发挥社区养老服务依托作用，强化机构养老专业支撑能力，创新普惠养老服务模式，加快补齐农村养老服务短板。科学谋划布局，强化经营主体引育，大力支持推动养老产业高质量发展。深入开展新时代“三晋银龄行动”，构建老年友好型社会。二要全力做好社会救助工作，切实兜牢民生底线。深化社会救助制度改革创新，完善分层分类社会救助体系，强化动态监测预警，推动社会救助向“物质+服务”综合救助转变，用心用情做好社会福利工作。三要着力优化社会事务服务，持续提升基本社会服务水平。优化婚姻登记管理服务，推动跨区域通办，构建新型婚育文化。深化殡葬改革，加快补齐殡葬领域公共服务设施短板。四要加强基层社会治理，不断提升社会治理效能。坚持和发展新时代“枫桥经验”，强化城乡社区治理，健全网格化管理机制，推进智慧社区建设。加强社会组织登记管理、综合监管。规范区划地名管理，传承弘扬地名文化。创新慈善服务模式，促进慈善事业发展。五要不折不扣狠抓落实，确保各项工作落地见效。加强组织领导，谋实民政项目和政策举措，走好新时代群众路线，以“时时放心

In [17]:
doc0 = {
    "dataTime":docs[0]["dataTime"],
    "docContent":docs[0]["contentText"],
    "title":docs[0]["title"],
    "docEmbedding":None,
    
}

In [44]:
add_doc(es_tool, "test", doc0, id= "0")

Index 'test' ,  Doc Id '0' ADD successfully.


True

In [2]:
# 写入全部数据到数据库

import torch
import torch.nn.functional as F

from torch import Tensor
from modelscope import AutoTokenizer, AutoModel
import numpy as np

In [3]:
def last_token_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

In [4]:
tokenizer = AutoTokenizer.from_pretrained("/data/zhengwj/model/Qwen/Qwen/Qwen3-Embedding-0.6B",  padding_side='left')
model = AutoModel.from_pretrained("/data/zhengwj/model/Qwen/Qwen/Qwen3-Embedding-0.6B")

In [5]:
import math
def batch_embedding(tokenizer, model, texts, batch_size, max_length):
    count = len(texts)
    results = []
    print(f"all Emebedding texts :{count} items")
    for i in range(math.ceil(count/batch_size) ):
        print(f"Start to embedding:{i*batch_size}_{(i+1)*batch_size}")
        input_texts = texts[i*batch_size:(i+1)*batch_size]
        batch_dict = tokenizer(
            input_texts,
            padding=True,
            truncation=True,
            max_length = max_length,
            return_tensors="pt"
        )
        batch_dict.to(model.device)
        outputs = model(**batch_dict)
        embeddings = last_token_pool(outputs.last_hidden_state, batch_dict["attention_mask"])

        embeddings = F.normalize(embeddings, p=2, dim=1).detach().numpy()
        results.extend(embeddings)
    return results

In [20]:
embeddings = batch_embedding(tokenizer, model, [d["contentText"] for d in docs[0:32]], 16, 8192)

all Emebedding texts :32 items
Start to embedding:0_16
Start to embedding:16_32


In [24]:
for i in range(32):
    document = {
        "dataTime":docs[i]["dataTime"],
        "docContent":docs[i]["contentText"],
        "title":docs[i]["title"],
        "docEmbedding":[1]*1536,  
    }
    add_doc(es_tool, "test", doc0, id= f"test : {i}")

Index 'test' ,  Doc Id 'test : 0' ADD successfully.
Index 'test' ,  Doc Id 'test : 1' ADD successfully.
Index 'test' ,  Doc Id 'test : 2' ADD successfully.
Index 'test' ,  Doc Id 'test : 3' ADD successfully.
Index 'test' ,  Doc Id 'test : 4' ADD successfully.
Index 'test' ,  Doc Id 'test : 5' ADD successfully.
Index 'test' ,  Doc Id 'test : 6' ADD successfully.
Index 'test' ,  Doc Id 'test : 7' ADD successfully.
Index 'test' ,  Doc Id 'test : 8' ADD successfully.
Index 'test' ,  Doc Id 'test : 9' ADD successfully.
Index 'test' ,  Doc Id 'test : 10' ADD successfully.
Index 'test' ,  Doc Id 'test : 11' ADD successfully.
Index 'test' ,  Doc Id 'test : 12' ADD successfully.
Index 'test' ,  Doc Id 'test : 13' ADD successfully.
Index 'test' ,  Doc Id 'test : 14' ADD successfully.
Index 'test' ,  Doc Id 'test : 15' ADD successfully.
Index 'test' ,  Doc Id 'test : 16' ADD successfully.
Index 'test' ,  Doc Id 'test : 17' ADD successfully.
Index 'test' ,  Doc Id 'test : 18' ADD successfully.
Ind

In [22]:
embeddings[0].size

1024

In [ ]:
dsl = {
            "query": {
                "bool": {
                    "must": [
                    ],
                    "should": [
                    ],
                    "must_not": [
                    ],
                    "minimum_should_match": 1,
                    "boost": 1.0
                }
            },
            "size":30
}

In [ ]:
should_list = [
    {"match_phrase": {"docContext": {"query": "供销合作社", "boost": 1}}}, 
]
dsl["query"]["bool"]["should"] = should_list

In [ ]:
es_tool.search(index="gov", body=dsl)["hits"]["hits"]

In [ ]:
dsl = {
            "query": {
                "bool": {
                    "must": [
                        {
                            "knn": {
                                "field": "docEmbedding",
                                "query_vector": []
                            }
                        }
                    ],
                    "should": [
                    ],
                    "minimum_should_match": 0,
                    "boost": 1.0
                }
            },
            "size": 30
        }

In [ ]:
query = "省政府党组第36次会议暨省政府第61次常务会议"
task = 'Given a web search query, retrieve relevant passages that answer the query'
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery:{query}'
embeddings = batch_embedding(tokenizer, model, [get_detailed_instruct(task, query)], 16, 8192)

In [ ]:
dsl["query"]["bool"]["must"][0]["knn"]["query_vector"] = embeddings[0]
es_tool.search(index="gov", body=dsl)["hits"]["hits"]
